<a href="https://colab.research.google.com/github/TPS-Projects/Colab/blob/main/Extra%C3%A7%C3%A3o_de_Extratos_Crehnor_Cresol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Apagar Pastas

# !rm -rf /content/_ /

# Apagar arquivos da pasta
'''
!rm -rf /content/_Cresol/*
!rm -rf /content/_Crehnor/*
'''

In [4]:
# Cria as pastas
import os

pasta1 = '/content/_Crehnor'
pasta2 = '/content/_Cresol'

os.makedirs(pasta1, exist_ok=True)
os.makedirs(pasta2, exist_ok=True)

# Verifica se ambas foram criadas e exibe a mensagem correspondente
if os.path.isdir(pasta1) and os.path.isdir(pasta2):
    print(f"As pastas {pasta1}\n- {pasta2} foram criadas:\n- Faça o uploado dos arquivos em pdf")
else:
    print("Erro ao criar uma ou mais pastas.")

As pastas /content/_Crehnor
- /content/_Cresol foram criadas:
- Faça o uploado dos arquivos em pdf


In [ ]:
#instalar pyplumber
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 660.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 17.4 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
# 1. Instala as bibliotecas necessárias no Colab.
#criar a pasta

import os
import re
import pdfplumber
import pandas as pd


# 2. COLE O CAMINHO DA SUA PASTA AQUI
pasta_pdfs = "/content/_Crehnor"

dados_extraidos = []

print("Iniciando a leitura dos PDFs...")

for arquivo in os.listdir(pasta_pdfs):
    if arquivo.lower().endswith(".pdf"):
        caminho_completo = os.path.join(pasta_pdfs, arquivo)
        print(f"Lendo: {arquivo}")

        try:
            with pdfplumber.open(caminho_completo) as pdf:
                texto = ""
                for pagina in pdf.pages:
                    texto += pagina.extract_text() + "\n"

                dados = {
                    "Arquivo": arquivo,
                    "Agência": None,
                    "Conta": None,
                    "Cliente/Nome": None,
                    "CNPJ": None,
                    "Saldo": None,
                    "Tipo Layout": None
                }

                # Layout 1: Conta Bloqueada / Corrente
                if "SALDO DISPONÍVEL" in texto:
                    dados["Tipo Layout"] = "Conta Corrente"

                    agencia = re.search(r"AGÊNCIA:\s*(.*)", texto)
                    conta = re.search(r"CONTA:\s*(\d+)", texto)
                    cliente = re.search(r"CLIENTE:\s*(.*)", texto)
                    cnpj = re.search(r"CPF/CNPJ:\s*([\d\.\-\/]+)", texto)

                    # REGEX ATUALIZADO: Aceita quebras de linha e barras invisíveis
                    saldo = re.search(r"SALDO DISPONÍVEL \(\=\)[\s\|]*(R\$\s*[\d\.,]+)", texto)

                    if agencia: dados["Agência"] = agencia.group(1).strip()
                    if conta: dados["Conta"] = conta.group(1).strip()
                    if cliente: dados["Cliente/Nome"] = cliente.group(1).strip()
                    if cnpj: dados["CNPJ"] = cnpj.group(1).strip()
                    if saldo: dados["Saldo"] = saldo.group(1).strip()

                # Layout 2: Extrato de Aplicação
                elif "AG/CONTA:" in texto and "NOME:" in texto:
                    dados["Tipo Layout"] = "Aplicação"

                    ag_conta = re.search(r"AG/CONTA:\s*(\d+)/(\d+)", texto)
                    nome = re.search(r"NOME:\s*(.*)", texto)
                    cnpj = re.search(r"CNPJ:\s*([\d\.\-\/]+)", texto)

                    # REGEX ATUALIZADO: Aceita quebras de linha e barras invisíveis
                    total = re.search(r"Total[\s\|]*([\d\.,]+[A-Z]?)", texto)

                    if ag_conta:
                        dados["Agência"] = ag_conta.group(1).strip()
                        dados["Conta"] = ag_conta.group(2).strip()
                    if nome: dados["Cliente/Nome"] = nome.group(1).strip()
                    if cnpj: dados["CNPJ"] = cnpj.group(1).strip()
                    if total: dados["Saldo"] = total.group(1).strip()

                dados_extraidos.append(dados)
        except Exception as e:
            print(f"Erro ao ler o arquivo {arquivo}: {e}")

# 3. Gerar a planilha Excel
df = pd.DataFrame(dados_extraidos)
caminho_saida = "/content/Extratos_CREHNOR.xlsx"
df.to_excel(caminho_saida, index=False)

print("-" * 30)
print(f"Extração concluída com sucesso!")
print(f"Sua tabela foi salva em: {caminho_saida}")

In [ ]:
#Tesseract e bibliotecas
!apt-get update -qq
!apt-get install poppler-utils tesseract-ocr tesseract-ocr-por -qq
!pip install pdf2image pytesseract pandas openpyxl -q


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Selecting previously unselected package tesseract-ocr-por.
Preparing to unpa

In [ ]:
#Extração Cresol

import os
import re
import pandas as pd
from pdf2image import convert_from_path
import pytesseract

# Nome da Pasta já criada
pasta_pdfs = "/content/_Cresol"

dados_extraidos = []

print("Iniciando a extração visual (OCR) dos extratos Cresol...")

for arquivo in os.listdir(pasta_pdfs):
    if arquivo.lower().endswith(".pdf"):
        caminho_completo = os.path.join(pasta_pdfs, arquivo)

        try:
            # Converte o PDF em imagem (300 dpi para não perder qualidade nas letras)
            imagens = convert_from_path(caminho_completo, dpi=300)
            texto = ""

            # Lê o texto da imagem usando o idioma português ("por")
            for img in imagens:
                texto += pytesseract.image_to_string(img, lang="por") + "\n"

            # Processa apenas Cresol
            if "EXTRATO CONSOLIDADO DE POUPANÇA" in texto.upper() or "EXTRATO CONSOLIDADO DE CONTA" in texto.upper():
                print(f"Lendo com OCR: {arquivo}")

                dados = {
                    "Arquivo": arquivo,
                    "Agência": None,
                    "Conta": None,
                    "Cliente/Nome": None,
                    "CNPJ": None,
                    "Saldo": None,
                    "Tipo Layout": None
                }

# LAYOUT A: Poupança

                if "POUPANÇA" in texto.upper() or "POUPANCA" in texto.upper():
                    dados["Tipo Layout"] = "Poupança"

                    # No OCR, a linha fica limpa: "AGENCIA: 1762 0 CONECTA FRONTEIRAS"
                    agencia = re.search(r"AG[EÊ]NCIA:\s*(.*)", texto, re.IGNORECASE)
                    if agencia:
                        # Limpa qualquer "FONE:" que possa ter grudado no final
                        dados["Agência"] = agencia.group(1).split("FONE")[0].strip()

                    # A linha da conta também fica limpa
                    conta_nome = re.search(r"CONTA:\s*([\d\.\-]+)\s+(.*)", texto, re.IGNORECASE)
                    if conta_nome:
                        dados["Conta"] = conta_nome.group(1).strip()
                        dados["Cliente/Nome"] = conta_nome.group(2).strip()

                    # CNPJ (se houver)
                    cnpj = re.search(r"(\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2})", texto)
                    if cnpj:
                        dados["CNPJ"] = cnpj.group(1).strip()

                    # O saldo fica muito mais fácil de capturar visualmente
                    saldo = re.search(r"SALDO\s*TOTAL[^\d]*([\d\.,]+\s*[CD]?)", texto, re.IGNORECASE)
                    if saldo:
                        dados["Saldo"] = saldo.group(1).strip()

# LAYOUT B: Conta Corrente

                elif "CONTA CORRENTE" in texto.upper():
                    dados["Tipo Layout"] = "Conta Corrente"

                    cnpj = re.search(r"(\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2})", texto)

                    # Pega a conta e o nome separados por traço na mesma linha
                    conta_nome = re.search(r"([\d\.]+\-\d{1,2})[\s\-]+([A-Z][A-Z\s]+)", texto)

                    # Pega a agência logo abaixo do título
                    agencia = re.search(r"EXTRATO CONSOLIDADO DE CONTA CORRENTE[\s\n]*(\d{4}\-\d\s+[A-Z\s]+)", texto, re.IGNORECASE)

                    saldo = re.search(r"SALDO\s*TOTAL[^\d]*([\d\.,]+\s*[CD]?)", texto, re.IGNORECASE)

                    if agencia: dados["Agência"] = agencia.group(1).strip()
                    if conta_nome:
                        dados["Conta"] = conta_nome.group(1).strip()
                        dados["Cliente/Nome"] = conta_nome.group(2).strip()
                    if cnpj: dados["CNPJ"] = cnpj.group(1).strip()
                    if saldo: dados["Saldo"] = saldo.group(1).strip()

                dados_extraidos.append(dados)
        except Exception as e:
            print(f"Erro ao ler o arquivo {arquivo} com OCR: {e}")

# 3. Gerar a planilha Excel dedicada
df = pd.DataFrame(dados_extraidos)
caminho_saida = "/content/Extratos_Cresol.xlsx"
df.to_excel(caminho_saida, index=False)

print("-" * 30)
print(f"Extração visual concluída com sucesso!")
print(f"Sua tabela gerada por OCR foi salva em: {caminho_saida}")